In [62]:
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

dir = Path().resolve().parents[1]

if dir not in sys.path:
    print("directory path is not in the system path")
    sys.path.append(str(dir))
    print("adding directory...")
else:
    print("Directory already exists in the system path")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
directory path is not in the system path
adding directory...


In [97]:
from nn import Unet1D, Returns, RMSELoss
import torch
from torch.utils.data import DataLoader
from scripts import evaluate
import math
import yfinance as yf
from utils import log_transform, posterior_beta
import matplotlib.pyplot as plt
from diffusion import forward, reverse

In [64]:
ticker = "^GSPC"
start_interval = "2016-12-01"
end_interval = "2026-01-01"
interval = "1d"

raw_snp500 = torch.tensor(yf.Ticker(ticker).history(start=start_interval, end=end_interval, interval=interval)["Close"].to_numpy())

split = math.ceil(len(raw_snp500) * 0.15)
test_split = len(raw_snp500) - math.ceil(len(raw_snp500) * 0.15)
test_raw_snp500 = raw_snp500[test_split:]

window_size = 32

test_data = Returns(
  raw_returns=test_raw_snp500,
  window_size=window_size,
  transform=log_transform
)

print(len(test_data))

test_dataloader = DataLoader(test_data, batch_size=32, shuffle=True, drop_last=True)
print(next(iter(test_dataloader)))

311
tensor([[[-5.5174e-03, -2.8507e-03, -5.0216e-03,  ...,  1.7192e-03,
          -1.1806e-02,  3.6469e-03]],

        [[-7.1945e-05,  2.4420e-03,  2.3741e-03,  ..., -1.9935e-02,
           5.5232e-03,  3.7741e-03]],

        [[ 1.3601e-04, -1.5992e-02,  1.5730e-02,  ..., -1.5825e-02,
           9.0895e-02, -3.5221e-02]],

        ...,

        [[ 6.6901e-03,  3.3957e-04, -2.7276e-03,  ...,  1.7492e-02,
           1.5730e-03, -1.1220e-02]],

        [[ 2.4923e-03, -6.1632e-03, -2.9683e-03,  ...,  6.1192e-03,
           5.2994e-03, -2.8592e-03]],

        [[ 1.2558e-03,  2.6449e-04, -2.9006e-03,  ..., -2.9950e-04,
           2.6479e-03,  1.6128e-03]]])


In [65]:
encoder_in_channels = [1, 4, 8, 16]
encoder_out_channels = [4, 8, 16, 32]
decoder_in_channels = [32, 16, 8, 4]
decoder_out_channels = [16, 8, 4, 1]
attn_res = 16
n_res_block = 2
T = 1000
num_heads = 4
betas = torch.linspace(1e-4, 2e-2, T)
alpha_hats = torch.cumprod(
  input=1-betas,
  dim=0,
  dtype=torch.float32
)

model = Unet1D(
  attn_res=attn_res,
  n_res_block=n_res_block,
  encoder_in_channels=encoder_in_channels,
  encoder_out_channels=encoder_out_channels,
  decoder_in_channels=decoder_in_channels,
  decoder_out_channels=decoder_out_channels,
  T=T,
  num_heads=num_heads
)
loss_fn = RMSELoss()

In [66]:
SAVE_PATH = dir / "models" / "model_v0.pth"
SAVE_PATH.exists()

True

In [67]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device

'cpu'

In [68]:
checkpoint = torch.load(SAVE_PATH, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [71]:
test_result = evaluate(
  data=test_dataloader,
  loss_fn=loss_fn,
  model=model,
  alpha_hats=alpha_hats,
  T=T,
  device=device
)

test_result

0.3022476269139184

In [78]:
test_sample = next(iter(test_dataloader))

In [91]:
# quick sanity check
T = 1000
batch_size = 32

t = torch.randint(0, T, size=(batch_size,))
xt, _ = forward(test_sample, alpha_hats, t)

epsilon_theta = model(xt, t)
epsilon_theta.mean(), epsilon_theta.std()

(tensor(0.0295, grad_fn=<MeanBackward0>),
 tensor(0.8784, grad_fn=<StdBackward0>))

In [102]:
# reverse process
xT = torch.randn(size=(32, 1, 32))
posterior_betas = torch.tensor([posterior_beta(alpha_hats=alpha_hats, betas=betas, t=t) for t in range(T)])

x0 = reverse(
  xT=xT,
  T=T,
  betas=betas,
  posterior_betas=posterior_betas,
  alpha_bars=alpha_hats,
  model=model
)

In [104]:
x0

tensor([[[-0.3458,  0.6366, -0.5767,  ..., -0.0598, -0.1351, -0.2149]],

        [[-0.3304, -0.9564,  0.3708,  ..., -0.1486,  0.0022,  0.0339]],

        [[ 0.0741,  0.1556,  0.5399,  ..., -0.0553, -0.0741,  0.1024]],

        ...,

        [[-0.1696,  0.0288,  0.2576,  ...,  0.7368, -0.1202, -0.1330]],

        [[ 0.4489,  0.2223, -0.0717,  ...,  0.3061, -0.8533, -0.3806]],

        [[-0.1780, -0.0868,  0.3527,  ...,  0.3897,  0.7387,  0.3167]]],
       grad_fn=<AddBackward0>)